# Create Agent Builder Tools via Kibana API

This notebook creates the two Agent Builder tools used in the article
_Connecting Cursor to production logs via the Elastic MCP Server_:

- `get_filter_usage`: queries `traces-apm-default` for filter click counts
- `get_recent_errors`: queries `logs-apm.error-default` for recent error groups

**Requirements:** Elasticsearch 9.3+ or Elastic Cloud Serverless

Copy `.env.example` to `.env` and fill in your values before running.

In [ ]:
%pip install dotenv requests

In [ ]:
import os
import json
import requests
from dotenv import load_dotenv

load_dotenv()

KIBANA_URL = os.getenv("KIBANA_URL")
API_KEY = os.getenv("ELASTICSEARCH_API_KEY")

headers = {
    "Authorization": f"ApiKey {API_KEY}",
    "Content-Type": "application/json",
    "kbn-xsrf": "true",
}

print(f"Kibana URL: {KIBANA_URL}")

## Tool 1: get_filter_usage

Returns the usage count for each search filter, sorted by most used first.
Reads RUM interaction events from `traces-apm-default`.

In [ ]:
tool_filter_usage = {
    "id": "get_filter_usage",
    "type": "esql",
    "description": "Returns the usage count for each search filter in the ecommerce-search-ui service, sorted by most used first.",
    "tags": ["apm", "rum", "filters"],
    "configuration": {
        "query": (
            "FROM traces-apm-default\n"
            '| WHERE service.name == "ecommerce-search-ui"\n'
            '| WHERE transaction.type == "user-interaction"\n'
            "| WHERE labels.filter_name IS NOT NULL\n"
            "| STATS count = COUNT(*) BY labels.filter_name\n"
            "| SORT count DESC"
        ),
        "params": {},
    },
}

response = requests.post(
    f"{KIBANA_URL}/api/agent_builder/tools", headers=headers, json=tool_filter_usage
)

print(f"Status: {response.status_code}")
print(json.dumps(response.json(), indent=2))

## Tool 2: get_recent_errors

Returns the most frequent error groups for a service, ranked by occurrence count.
Reads from `logs-apm.error-default`.

In [ ]:
tool_recent_errors = {
    "id": "get_recent_errors",
    "type": "esql",
    "description": "Returns the most frequent error groups for ecommerce-search-ui, ranked by occurrence count, with the exception message and code location.",
    "tags": ["apm", "errors", "debugging"],
    "configuration": {
        "query": (
            """FROM logs-apm.error-default
                | WHERE service.name == "ecommerce-search-ui"
                | WHERE processor.name == "error"
                | STATS count = COUNT(*) BY error.grouping_key, error.exception.message, error.culprit
                | SORT count DESC
                | LIMIT 5
            """
        ),
        "params": {},
    },
}

response = requests.post(
    f"{KIBANA_URL}/api/agent_builder/tools", headers=headers, json=tool_recent_errors
)

print(f"Status: {response.status_code}")
print(json.dumps(response.json(), indent=2))

## Verify

List all Agent Builder tools to confirm both were created successfully.

In [ ]:
response = requests.get(f"{KIBANA_URL}/api/agent_builder/tools", headers=headers)

tools = response.json()
print(f"Total tools: {len(tools)}")
for tool in tools:
    print(f"  - {tool.get('id')}: {tool.get('description', '')[:60]}")

## Cleanup

Delete both tools from Agent Builder. Run this cell to remove them from your Kibana instance.

In [ ]:
tool_ids = ["get_filter_usage", "get_recent_errors"]

for tool_id in tool_ids:
    response = requests.delete(
        f"{KIBANA_URL}/api/agent_builder/tools/{tool_id}",
        headers=headers,
    )
    print(f"Deleted {tool_id}: {response.status_code}")